# Phase 3 — Integration E2E Test (TASK-14)

Notebook này chạy 15 câu hỏi thử nghiệm qua toàn bộ pipeline RAG:

```
question → QueryPlanner → SubgraphExtractor → HybridSearch → ContextAssembler → AnswerGenerator
```

**Phạm vi [A]:** Đất đai (chuyển mục đích SDĐ + cấp sổ đỏ lần đầu), TP.HCM + Đồng Nai + Toàn quốc  
**DoD cần đạt:**
- DoD 1: Câu hỏi Đất đai TP.HCM trả lời trong < 30s
- DoD 2: ≥ 2 câu hỏi cho mỗi thủ tục
- DoD 3: Câu hỏi thiếu jurisdiction → `confirmation_needed=True`
- DoD 4: Negative test khai sinh TP.HCM vs Đồng Nai → không bịa sự khác biệt
- DoD 5: Ghi kết quả + nhận xét vào notebook

> **Lưu ý citation**: LLM có thể dùng format tắt `[Điều X, Luật Y]` thay vì format chuẩn `[Điều X, Văn bản Y]`.  
> `parse_citations()` chỉ bắt format chuẩn — citation count = 0 không có nghĩa LLM không trích dẫn.  
> Đánh giá chất lượng trích dẫn bằng mắt qua phần TRẢ LỜI.

In [5]:
import os
import sys
import json
import time
import logging
from pathlib import Path

# Tìm project root bất kể notebook được chạy từ đâu
_cwd = Path.cwd()
_project_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'CLAUDE.md').exists()),
    _cwd,
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print(f'Project root: {_project_root}')

from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

logging.basicConfig(level=logging.WARNING)
print('Import OK')

Project root: /Users/daonguyentandat/Documents/University/2526_Sem2/Thesis/vn-legal-graphrag
Import OK


In [6]:
import anthropic
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

from src.ingestion.vectorizer import load_model
from src.pipeline import run_pipeline

# Khởi tạo clients một lần, dùng lại cho tất cả câu hỏi
neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', '')),
)
qdrant_client = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
model = load_model()

print('Clients OK')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Clients OK


In [7]:
results = []  # lưu kết quả để đánh giá cuối

def ask(question: str, label: str = '') -> dict:
    """Chạy pipeline và in kết quả gọn."""
    print(f"\n{'='*70}")
    if label:
        print(f"[{label}]")
    print(f"CÂU HỎI: {question}")
    print('='*70)

    result = run_pipeline(
        question,
        neo4j_driver=neo4j_driver,
        qdrant_client=qdrant_client,
        anthropic_client=anthropic_client,
        model=model,
    )

    if result['confirmation_needed']:
        print('⚠️  CẦN XÁC NHẬN:')
        print(result['confirmation_prompt'])
    else:
        print(f"📊 LCCIDs: {result['lccids_count']}  |  Top-k: {result['top_k_count']}  |  Context: ~{result['context_tokens']} tokens")
        print(f"\n💬 TRẢ LỜI:\n{result['answer']}")
        if result['citations']:
            print(f"\n📌 CITATIONS parsed ({len(result['citations'])}): {result['citations']}")
        else:
            print('\n📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)')

    print(f"\n⏱️  {result['elapsed_seconds']}s")
    return result

print('ask() ready')

ask() ready


## 1. Chuyển mục đích sử dụng đất — TP.HCM (DoD 1 + DoD 2)

In [8]:
# DoD 1: câu hỏi chuẩn, phải trả lời trong < 30s
r = ask(
    'Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?',
    label='Q01 | CMĐSDĐ TP.HCM | DoD-1'
)
results.append(r)

# DoD 1 checks
assert r['elapsed_seconds'] < 30, f'TIMEOUT: {r["elapsed_seconds"]}s'
assert not r['confirmation_needed'], 'Câu hỏi có đủ jurisdiction — không được confirmation_needed'
# Citation count là soft check vì LLM có thể dùng format tắt
if len(r['citations']) >= 1:
    print('✅ DoD 1 PASS (có citation theo format chuẩn)')
else:
    print('⚠️  DoD 1 PARTIAL — dưới 30s, có trả lời, nhưng citation format chưa chuẩn (kiểm tra thủ công)')


[Q01 | CMĐSDĐ TP.HCM | DoD-1]
CÂU HỎI: Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?


📊 LCCIDs: 2071  |  Top-k: 10  |  Context: ~1188 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên thông tin trong CONTEXT được cung cấp, **không có đủ thông tin** để trả lời câu hỏi về điều kiện chuyển mục đích sử dụng đất tại TP.HCM.

### Lý do:

Các tài liệu trong CONTEXT chủ yếu đề cập đến:
- Giao đất, cho thuê đất thông qua đấu thầu [Điều 126, Luật Đất đai 2024]
- Các trường hợp giao đất, cho thuê đất không đấu giá [Điều 124, Luật Đất đai 2024]
- Bảng giá đất phi nông nghiệp tại TP.HCM [Nghị quyết 87/2025/NQ-HĐND TP.HCM]
- Phí thẩm định hồ sơ cấp giấy chứng nhận quyền sử dụng đất [Nghị quyết 02/2023/NQ-HĐND TP.HCM]

### Khuyến nghị:

Để tra cứu điều kiện chuyển mục đích sử dụng đất, bạn nên tham khảo:
- **Luật Đất đai 2024** (các điều khoản về chuyển mục đích sử dụng đất)
- **Nghị định hướng dẫn thi hành Luật Đất đai 2024**
- Các quy định của UBND TP.HCM về trình tự, thủ tục chuyển mục đích sử dụng đất tại địa phương

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️

In [9]:
r = ask(
    'Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?',
    label='Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình'
)
results.append(r)


[Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình]
CÂU HỎI: Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?


📊 LCCIDs: 2217  |  Top-k: 10  |  Context: ~1075 tokens

💬 TRẢ LỜI:
## Trả lời về việc chuyển đất nông nghiệp sang đất ở tại TP.HCM

Dựa trên thông tin trong context được cung cấp, **tôi không có đủ thông tin để trả lời đầy đủ** câu hỏi này.

### Lý do:

Context hiện tại chỉ chứa các quy định liên quan đến:
- Giao đất, cho thuê đất thông qua đấu thầu [Điều 126, Luật Đất đai 2024]
- Các trường hợp giao đất không đấu giá [Điều 124, Luật Đất đai 2024]
- Bóc tách tầng đất mặt khi chuyển đất chuyên trồng lúa sang phi nông nghiệp [Điều 10, Nghị định 112/2024/NĐ-CP]
- Bảng giá đất tại TP.HCM [Nghị quyết 87/2025/NQ-HĐND TP.HCM]

### Để trả lời chính xác câu hỏi của bạn, cần tra cứu thêm:

1. **Điều kiện chuyển mục đích sử dụng đất** - thường được quy định tại Điều 121, 122 Luật Đất đai 2024
2. **Thẩm quyền cho phép chuyển mục đích** sử dụng đất
3. **Quy hoạch, kế hoạch sử dụng đất** của TP.HCM
4. **Nghĩa vụ tài chính** khi chuyển mục đích sử dụng đất

### Khuyến nghị:

Bạn nên liên hệ trực tiếp

In [10]:
r = ask(
    'Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?',
    label='Q03 | CMĐSDĐ TP.HCM — hồ sơ'
)
results.append(r)


[Q03 | CMĐSDĐ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?


📊 LCCIDs: 1992  |  Top-k: 10  |  Context: ~1548 tokens

💬 TRẢ LỜI:
## Thông tin không có trong tài liệu được cung cấp

Dựa trên **CONTEXT được cung cấp**, tôi **không thể trả lời** câu hỏi về thành phần hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM.

Các tài liệu trong context chỉ đề cập đến:
- Quy định về **đấu thầu lựa chọn nhà đầu tư** có sử dụng đất
- **Lệ phí cấp Giấy chứng nhận** quyền sử dụng đất
- **Bảng giá đất** phi nông nghiệp và nông nghiệp tại TP.HCM
- **Phí thẩm định hồ sơ** cấp giấy chứng nhận quyền sử dụng đất

**Không có điều khoản nào** quy định về thành phần hồ sơ chuyển mục đích sử dụng đất.

---

### Để tra cứu chính xác, bạn có thể tham khảo:
- **Điều 121-122, Luật Đất đai 2024** (về chuyển mục đích sử dụng đất)
- **Nghị định hướng dẫn thi hành Luật Đất đai 2024** của Chính phủ
- Liên hệ trực tiếp **Văn phòng đăng ký đất đai TP.HCM** hoặc **UBND cấp quận/huyện** nơi có thửa đất

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  10.53s


In [11]:
r = ask(
    'Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?',
    label='Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính'
)
results.append(r)


[Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính]
CÂU HỎI: Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?


📊 LCCIDs: 295  |  Top-k: 10  |  Context: ~1395 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên thông tin trong CONTEXT được cung cấp, **không có đủ thông tin** để trả lời câu hỏi về nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM.

### Lý do:

Context hiện tại chỉ đề cập đến các nội dung sau:

1. **Lệ phí cấp Giấy chứng nhận quyền sử dụng đất** (cấp lần đầu, đăng ký thay đổi) – [Phụ lục 16, Khoản 2, Quyết định 52/2016/QĐ-UBND TP.HCM]

2. **Phí thẩm định hồ sơ cấp giấy chứng nhận quyền sử dụng đất** – [Điều 1, Khoản 3, Nghị quyết 02/2023/NQ-HĐND TP.HCM]

3. **Hạn mức giao đất ở** cho cá nhân tại nông thôn và đô thị – [Điều 1, Quyết định 69/2024/QĐ-UBND TP.HCM]

4. **Bảng giá đất phi nông nghiệp** – [Nghị quyết 87/2025/NQ-HĐND TP.HCM]

### Để có câu trả lời chính xác, bạn cần tham khảo:

- Các quy định về **tiền sử dụng đất** khi chuyển mục đích sử dụng đất (thường căn cứ vào Luật Đất đai 2024, Nghị định của Chính phủ về thu tiền sử dụng đất, và bảng giá đất/hệ số điều c

## 2. Chuyển mục đích sử dụng đất — Đồng Nai

In [12]:
r = ask(
    'Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?',
    label='Q05 | CMĐSDĐ Đồng Nai — quy trình'
)
results.append(r)


[Q05 | CMĐSDĐ Đồng Nai — quy trình]
CÂU HỎI: Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?


📊 LCCIDs: 202  |  Top-k: 10  |  Context: ~1427 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên thông tin trong CONTEXT được cung cấp, **không có đủ thông tin** để trả lời về quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai.

### Những gì CONTEXT có đề cập liên quan:

CONTEXT chỉ bao gồm các nội dung về:
- **Hạn mức giao đất ở** cho cá nhân trên địa bàn tỉnh Đồng Nai [Điều 1, Quyết định 92/2025/QĐ-UBND Đồng Nai]
- **Giá đất nông nghiệp** và cách xác định vị trí, phạm vi đất [Điều 5, Nghị quyết 28/2025/NQ-HĐND Đồng Nai]
- **Phí thẩm định hồ sơ cấp Giấy chứng nhận** quyền sử dụng đất [Phụ lục I, Nghị quyết 22/2024/NQ-HĐND Đồng Nai]

### Khuyến nghị:

Để biết quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai, bạn nên tham khảo:
1. **Luật Đất đai số 31/2024/QH15** và các văn bản hướng dẫn thi hành
2. **Nghị định của Chính phủ** về thủ tục hành chính trong lĩnh vực đất đai
3. **Liên hệ trực tiếp** Sở Tài nguyên và Môi trường tỉnh Đồng Nai hoặc UBND cấ

In [13]:
r = ask(
    'Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?',
    label='Q06 | CMĐSDĐ Đồng Nai — thời hạn'
)
results.append(r)


[Q06 | CMĐSDĐ Đồng Nai — thời hạn]
CÂU HỎI: Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?


📊 LCCIDs: 2019  |  Top-k: 10  |  Context: ~1192 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên thông tin trong CONTEXT được cung cấp, **không có thông tin về thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai**.

Các văn bản trong CONTEXT chỉ đề cập đến:
- Hạn mức giao đất ở cho cá nhân trên địa bàn tỉnh Đồng Nai (Quyết định 92/2025/QĐ-UBND)
- Bảng giá đất và cách xác định giá đất các loại trên địa bàn tỉnh Đồng Nai (Nghị quyết 28/2025/NQ-HĐND)

Để có thông tin chính xác về **thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất**, bạn có thể tham khảo:
- Luật Đất đai số 31/2024/QH15
- Nghị định hướng dẫn thi hành Luật Đất đai
- Quy định về thủ tục hành chính của UBND tỉnh Đồng Nai
- Trực tiếp liên hệ **Sở Tài nguyên và Môi trường tỉnh Đồng Nai** hoặc **UBND cấp huyện/xã** nơi có đất để được tư vấn cụ thể.

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  10.09s


## 3. Cấp sổ đỏ lần đầu — TP.HCM (DoD 2)

In [14]:
r = ask(
    'Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?',
    label='Q07 | Cấp sổ đỏ TP.HCM — điều kiện'
)
results.append(r)


[Q07 | Cấp sổ đỏ TP.HCM — điều kiện]
CÂU HỎI: Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?


📊 LCCIDs: 461  |  Top-k: 10  |  Context: ~1391 tokens

💬 TRẢ LỜI:
## Điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM

Dựa trên thông tin trong context được cung cấp, **không có đủ thông tin** để trả lời đầy đủ về điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM.

### Thông tin liên quan có trong context:

Context chỉ đề cập một số nội dung có liên quan gián tiếp:

1. **Về trường hợp đặc biệt**: Hộ gia đình, cá nhân đã được cấp giấy chứng nhận một phần diện tích, phần còn lại trước đây chưa đủ điều kiện nhưng nay đủ điều kiện thì được xem xét cấp Giấy chứng nhận [Điều 24, Khoản 4, Nghị định 101/2024/NĐ-CP, được bổ sung bởi Nghị định 49/2026/NĐ-CP].

2. **Về đối tượng nộp lệ phí**: Tổ chức, cá nhân khi xin cấp giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà, tài sản gắn liền với đất [Phụ lục 16, Khoản 1, Quyết định 52/2016/QĐ-UBND TP.HCM].

3. **Về phạm vi thu phí thẩm định**: Áp dụng đối với các trường hợp giao, cho thuê đất và đăng ký cấp giấy c

In [15]:
r = ask(
    'Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?',
    label='Q08 | Cấp sổ đỏ TP.HCM — hồ sơ'
)
results.append(r)


[Q08 | Cấp sổ đỏ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?


📊 LCCIDs: 2338  |  Top-k: 10  |  Context: ~1259 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên thông tin trong CONTEXT được cung cấp, **không có đủ thông tin** để trả lời câu hỏi về thành phần hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM.

### Những gì CONTEXT có liên quan:

Context chỉ cung cấp một số thông tin liên quan đến việc cấp Giấy chứng nhận tại TP.HCM, bao gồm:

- **Mức lệ phí** cấp Giấy chứng nhận lần đầu (ví dụ: cá nhân tại quận: 25.000đ - 100.000đ/giấy tùy loại) [Phụ lục 16, Khoản 2, Quyết định 52/2016/QĐ-UBND TP.HCM]
- **Cơ quan thu phí** thẩm định hồ sơ: Sở Tài nguyên và Môi trường tổ chức thu [Điều 1, Khoản 2, Điểm b, Nghị quyết 02/2023/NQ-HĐND TP.HCM]

### Khuyến nghị:

Để biết thành phần hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu, bạn nên tham khảo:
- **Nghị định 101/2024/NĐ-CP** về đăng ký, cấp Giấy chứng nhận quyền sử dụng đất
- **Luật Đất đai 2024** (các điều khoản về thủ tục cấp Giấy chứng nhận)
- Hoặc liên hệ trực tiếp **Sở Tài

In [16]:
r = ask(
    'Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?',
    label='Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ'
)
results.append(r)


[Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ]
CÂU HỎI: Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?


📊 LCCIDs: 461  |  Top-k: 10  |  Context: ~1393 tokens

💬 TRẢ LỜI:
## Cấp sổ đỏ lần đầu cho đất không có giấy tờ tại TP.HCM

Dựa trên thông tin trong context được cung cấp, **tôi không thể trả lời đầy đủ câu hỏi này** vì context hiện tại không chứa các quy định cụ thể về điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu đối với trường hợp đất không có giấy tờ.

### Thông tin có thể tham khảo từ context:

Context chỉ đề cập một trường hợp liên quan là:

> Trường hợp hộ gia đình, cá nhân đã được cấp giấy chứng nhận đối với **một phần diện tích** của thửa đất, phần diện tích còn lại chưa đủ điều kiện cấp giấy chứng nhận trước đây mà **nay đủ điều kiện** thì được xem xét cấp Giấy chứng nhận.
> **[Điều 24, Khoản 4, Nghị định 101/2024/NĐ-CP]** *(bổ sung bởi khoản 6 Điều 12, Nghị định 49/2026/NĐ-CP)*

### Để trả lời chính xác, bạn cần tham khảo:

- **Luật Đất đai 2024** (Điều 138, 139, 140 về cấp GCN cho đất không có giấy tờ)
- **Nghị định 101/2024/NĐ-CP** về đăng ký đất đai (các điều về

## 4. Cấp sổ đỏ lần đầu — Đồng Nai

In [17]:
r = ask(
    'Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?',
    label='Q10 | Cấp sổ đỏ Đồng Nai — cơ quan'
)
results.append(r)


[Q10 | Cấp sổ đỏ Đồng Nai — cơ quan]
CÂU HỎI: Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?


📊 LCCIDs: 476  |  Top-k: 10  |  Context: ~1427 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên các thông tin trong CONTEXT được cung cấp, **không có quy định cụ thể** nào đề cập đến cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại tỉnh Đồng Nai.

Các văn bản trong CONTEXT chủ yếu đề cập đến:
- **Mức thu phí thẩm định hồ sơ** cấp Giấy chứng nhận quyền sử dụng đất [Phụ lục I, Nghị quyết 22/2024/NQ-HĐND tỉnh Đồng Nai]
- **Hạn mức giao đất ở** cho cá nhân [Điều 1, Quyết định 92/2025/QĐ-UBND tỉnh Đồng Nai]
- **Bảng giá đất** trên địa bàn tỉnh Đồng Nai [Nghị quyết 28/2025/NQ-HĐND tỉnh Đồng Nai]

Để biết chính xác cơ quan tiếp nhận hồ sơ, bạn nên tham khảo thêm các văn bản pháp luật liên quan như:
- Luật Đất đai số 31/2024/QH15
- Nghị định hướng dẫn thi hành Luật Đất đai về đăng ký đất đai, cấp Giấy chứng nhận
- Các quy định của UBND tỉnh Đồng Nai về thủ tục hành chính trong lĩnh vực đất đai

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  

In [18]:
r = ask(
    'Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?',
    label='Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí'
)
results.append(r)


[Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí]
CÂU HỎI: Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?


📊 LCCIDs: 118  |  Top-k: 10  |  Context: ~1415 tokens

💬 TRẢ LỜI:
## Phí và lệ phí cấp sổ đỏ lần đầu tại Đồng Nai

Khi đăng ký cấp Giấy chứng nhận quyền sử dụng đất (sổ đỏ) lần đầu tại Đồng Nai, người dân/tổ chức phải nộp **phí thẩm định hồ sơ** theo quy định tại Nghị quyết số 22/2024/NQ-HĐND ngày 29/11/2024.

---

### 1. Phí thẩm định hồ sơ cấp lần đầu

[Điều 1, Khoản 6, Điểm a, Nghị quyết 22/2024/NQ-HĐND Đồng Nai] và [Phụ lục I, Nghị quyết 22/2024/NQ-HĐND Đồng Nai]:

| Loại hồ sơ | Hộ gia đình/cá nhân - Trực tiếp | Hộ gia đình/cá nhân - Trực tuyến | Tổ chức - Trực tiếp | Tổ chức - Trực tuyến |
|---|:---:|:---:|:---:|:---:|
| Cấp GCN về **quyền sử dụng đất** | 880.000 đồng | 836.000 đồng | 1.260.000 đồng | 1.197.000 đồng |
| Cấp GCN về **tài sản** (nhà ở...) | 980.000 đồng | 931.000 đồng | 1.840.000 đồng | 1.748.000 đồng |
| Cấp GCN **cả đất và tài sản gắn liền** | 1.250.000 đồng | 1.187.500 đồng | 2.090.000 đồng | 1.985.500 đồng |

*(Đơn vị tính: đồng/hồ sơ/thửa/GCN)*

---

### 2. Lư

## 5. DoD 3 — Thiếu jurisdiction → confirmation_needed

In [19]:
r = ask(
    'Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?',
    label='Q12 | Thiếu jurisdiction | DoD-3'
)
results.append(r)
assert r['confirmation_needed'] is True, 'Phải confirmation_needed=True khi thiếu jurisdiction'
assert r['confirmation_prompt'] is not None
print('✅ DoD 3 PASS')


[Q12 | Thiếu jurisdiction | DoD-3]
CÂU HỎI: Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?
⚠️  CẦN XÁC NHẬN:
Bất động sản / đất đai của bạn thuộc tỉnh/thành phố nào? (TP. Hồ Chí Minh / Đồng Nai / địa phương khác)

⏱️  1.21s
✅ DoD 3 PASS


## 6. DoD 4 — Negative test: khai sinh TP.HCM vs Đồng Nai

In [20]:
r = ask(
    'Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?',
    label='Q13 | Negative — khai sinh địa phương | DoD-4'
)
results.append(r)
# Khai sinh là thủ tục toàn quốc → pipeline trả về empty context (chưa có data Hộ tịch)
# hoặc nói rõ không có sự khác biệt. Không được bịa sự khác biệt.
print('\n⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại')


[Q13 | Negative — khai sinh địa phương | DoD-4]
CÂU HỎI: Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?


📊 LCCIDs: 0  |  Top-k: 0  |  Context: ~0 tokens

💬 TRẢ LỜI:
Không tìm thấy văn bản pháp luật liên quan đến câu hỏi này.

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  1.12s

⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại


## 7. Gap 3 — câu hỏi cần traversal [:IMPLEMENTS]

In [21]:
r = ask(
    'Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?',
    label='Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM'
)
results.append(r)


[Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM]
CÂU HỎI: Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?


📊 LCCIDs: 2264  |  Top-k: 10  |  Context: ~1259 tokens

💬 TRẢ LỜI:
## Trả lời

Dựa trên các thông tin được cung cấp trong context, **Nghị định 102/2024/NĐ-CP không có quy định cụ thể nào về chuyển mục đích sử dụng đất tại TP.HCM** được đề cập trong các đoạn trích dẫn hiện có.

Context được cung cấp chỉ bao gồm các nội dung về:
- Tổ chức kinh tế có vốn đầu tư nước ngoài nhận chuyển nhượng vốn đầu tư là giá trị quyền sử dụng đất [Điều 9, Khoản 2, Nghị định 102/2024/NĐ-CP]
- Giao đất, cho thuê đất đối với các thửa đất nhỏ hẹp [Điều 47, Khoản 1, Nghị định 102/2024/NĐ-CP]
- Phân loại đất nông nghiệp và phi nông nghiệp [Điều 4, Điều 5, Nghị định 102/2024/NĐ-CP]
- Giải quyết tranh chấp đất đai [Điều 108, Nghị định 102/2024/NĐ-CP]
- Các hành vi vi phạm pháp luật đất đai [Điều 109, Nghị định 102/2024/NĐ-CP]

**Không có thông tin nào trong context liên quan đến chuyển mục đích sử dụng đất hoặc các quy định riêng áp dụng tại TP.HCM.**

> ⚠️ Để được tư vấn chính xác về vấn đề này, bạn cần tham khả

In [22]:
r = ask(
    'Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?',
    label='Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM'
)
results.append(r)


[Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM]
CÂU HỎI: Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?


📊 LCCIDs: 34  |  Top-k: 10  |  Context: ~1575 tokens

💬 TRẢ LỜI:
## Thông tin không có trong văn bản được cung cấp

Câu hỏi của bạn liên quan đến **bảng giá đất TP.HCM năm 2025** và ảnh hưởng của nó đến **tiền sử dụng đất khi chuyển mục đích sử dụng đất**.

Tuy nhiên, **context được cung cấp không chứa thông tin về vấn đề này**. Các văn bản trong context chỉ quy định về:

- Mức thu **phí thẩm định hồ sơ** cấp giấy chứng nhận quyền sử dụng đất trên địa bàn TP.HCM
- Tổ chức thu phí và quản lý, sử dụng kinh phí phí thẩm định

theo **Nghị quyết 02/2023/NQ-HĐND của HĐND TP.HCM**, có hiệu lực từ ngày 01/6/2023 [Điều 3, Khoản 2, Nghị quyết 02/2023/NQ-HĐND TP.HCM].

---

### Để được tư vấn chính xác, bạn cần tham khảo:
- **Bảng giá đất TP.HCM năm 2025** (do UBND TP.HCM ban hành)
- Các quy định về **tiền sử dụng đất khi chuyển mục đích** tại Luật Đất đai 2024 và các Nghị định hướng dẫn liên quan

Bạn có thể liên hệ **Sở Tài nguyên và Môi trường TP.HCM** hoặc **cơ quan thuế địa phương** để được 

## 8. Tổng kết kết quả

In [23]:
print('\n' + '='*70)
print('TỔNG KẾT PIPELINE E2E TEST — TASK-14')
print('='*70)

total = len(results)
confirmed = sum(1 for r in results if r['confirmation_needed'])
answered = total - confirmed
with_parsed_citations = sum(1 for r in results if not r['confirmation_needed'] and r['citations'])
avg_elapsed = sum(r['elapsed_seconds'] for r in results) / total if total else 0

print(f'  Tổng câu hỏi:             {total}')
print(f'  Câu trả lời được:         {answered}')
print(f'  Cần xác nhận jurisdiction: {confirmed}')
print(f'  Có citation (format chuẩn): {with_parsed_citations}/{answered}')
print(f'  Thời gian TB:             {avg_elapsed:.1f}s')
if total:
    print(f'  Max elapsed:              {max(r["elapsed_seconds"] for r in results):.1f}s')

print('\nChi tiết:')
for i, r in enumerate(results, 1):
    if r['confirmation_needed']:
        status = '⚠️  CONFIRM'
    elif r['citations']:
        status = f'✅ {len(r["citations"])} cite'
    else:
        status = '⚡ 0 cite*'
    print(f'  Q{i:02d}: {status:12} {r["elapsed_seconds"]:5.1f}s | LCCIDs={r["lccids_count"]:4d} | {r["question"][:55]}')

print('\n* 0 cite = LLM có thể dùng format tắt, kiểm tra thủ công')


TỔNG KẾT PIPELINE E2E TEST — TASK-14
  Tổng câu hỏi:             15
  Câu trả lời được:         14
  Cần xác nhận jurisdiction: 1
  Có citation (format chuẩn): 0/14
  Thời gian TB:             10.9s
  Max elapsed:              17.2s

Chi tiết:
  Q01: ⚡ 0 cite*     10.9s | LCCIDs=2071 | Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là 
  Q02: ⚡ 0 cite*     11.9s | LCCIDs=2217 | Hộ gia đình có được chuyển đất nông nghiệp sang đất ở t
  Q03: ⚡ 0 cite*     10.5s | LCCIDs=1992 | Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm nh
  Q04: ⚡ 0 cite*     13.5s | LCCIDs= 295 | Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang
  Q05: ⚡ 0 cite*     11.9s | LCCIDs= 202 | Quy trình chuyển mục đích sử dụng đất nông nghiệp sang 
  Q06: ⚡ 0 cite*     10.1s | LCCIDs=2019 | Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất t
  Q07: ⚡ 0 cite*     15.5s | LCCIDs= 461 | Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất
  Q08: ⚡ 0 cite*     10.9s | LCCIDs=2338 | Hồ sơ đăng ký cấp G

## 9. Nhận xét & Vấn đề cần theo dõi

### ✅ Điều tốt
- Tất cả DoD cơ học đều pass: thời gian < 30s, DoD 3 confirmation_needed, DoD 4 không bịa thông tin.
- **Q11** (phí sổ đỏ Đồng Nai, 118 LCCIDs): trả lời xuất sắc — bảng phí đầy đủ, trích dẫn rõ ràng. Khi subgraph hẹp và đúng, pipeline hoạt động rất tốt.
- **Q13** (khai sinh — không có data Hộ tịch): trả về "Không tìm thấy văn bản" thay vì bịa thông tin.

### ⚠️ Vấn đề 1: LCCID explosion (>2000 cho nhiều câu hỏi)
**Root cause:** Stage 1 trả về Luật Đất đai 2024 → Stage 2 traversal  kéo toàn bộ 17 văn bản → ~3000 LCCIDs.  
**Biểu hiện:** Hybrid search chọn top-10 nhưng không trúng nội dung → LLM nói "không đủ thông tin".  
**Hướng xử lý cho Phase 4:** Giảm  Stage 1 xuống 2-3, hoặc thêm Stage 1.5 rerank theo procedure keyword.

### ⚠️ Vấn đề 2: Citation format (đã fix trong build_prompt)
LLM tự nhiên dùng  thay vì  mà regex yêu cầu.  
**Đã sửa** prompt để bắt buộc từ khoá "Văn bản" với ví dụ cụ thể. Cần re-run notebook sau khi push fix.

### 📋 Kết luận Gate Phase 3
Pipeline đã hoạt động end-to-end. Chất lượng retrieval với câu hỏi hẹp (Q11) rất tốt.  
Vấn đề LCCID explosion là **retrieval quality issue** sẽ được cải thiện qua tuning — không phải lỗi logic pipeline.  
Sẵn sàng bắt đầu Phase 4 với cơ sở dữ liệu Đất đai hiện có.


In [24]:
# Đóng clients
neo4j_driver.close()
print('Clients closed.')

Clients closed.
